In [ ]:
"""
:Author: Qingcheng Wei
Date: 03/04/2026


Objective:
    Download SPP Mid-Term Load Forecast (MTLF) data.
    Parse ONLY the LATEST downloaded CSV file and update the database tables:

    Intermediate table:
    - spp_physical.MTLF                              (added baa_zone; PK: ForecastTime, IntervalEnd, baa_zone)

    Per-BAA output tables:
    - spp_physical.baa_zonal_latest_load_forecast    (latest forecast per BAA zone; PK: dt, hr, baa_zone)
    - spp_physical.baa_zonal_virtual_load_forecast   (virtual forecast per BAA zone; PK: dt, hr, baa_zone)

    Market-total output tables:
    - spp_physical.spp_latest_load_forecast          (latest forecast summed across all BAA zones; PK: dt, hr)
    - spp_physical.spp_virtual_load_forecast         (virtual forecast summed across all BAA zones; PK: dt, hr)

    Includes a backfill mechanism to check for and fill missing data from the last 20 days.

Data Source: ftp://pubftp.spp.org/Operational_Data/MTLF/

Calling this program:
    python spp_dataDownload_MTLF.py [START_DATE] [END_DATE]
    If dates are not provided, it defaults to yesterday and today.

Update every hour
"""

import sys
import os
import glob
import pandas as pd
import datetime
import pytz
import subprocess
import shutil

# Add project path for custom modules
sys.path.append("/var/www/python/Prod/nighthawk/")
from nighthawk.util import connections, sql_functions

sys.path.append("/var/www/python/Prod/COMMON/monitoring/notification_system/")
from send_msg_through_slack import send_msg_through_slack

script_name='spp_dataDownload_MTLF.py'

ESSENTIAL_ACTUAL_COLUMNS = {
    'Interval',
    'GMTIntervalEnd',
    'MTLF',
    'Averaged Actual'
}

# When BAA column is present, map its value to the baa_zone code stored in the DB.
# Old-format files that lack BAA are treated as SPP (East).
BAA_ZONE_MAP = {
    'SPP':  'E',
    'SWPW': 'W',
}

def log_message(level, type, line, message):
    print(f"{level} - {type} - Line {line}: {message}")


def download_forecast(dt):

    try:
        download_folder = "/var/www/python/Qingcheng/load/"

        if not os.path.exists(download_folder):
            try:
                os.makedirs(download_folder)
            except OSError as e:
                print(f"Error creating directory {download_folder}: {e}")
                return

        dt_str = dt.strftime("%Y/%m/%d")
        ftp_url = f"ftp://pubftp-mte.itespp.org/Operational_Data/MTLF/{dt_str}"

        cmd = f"wget -nd -r -nc {ftp_url} -P {download_folder}"
        print(f"Executing: {cmd}")
        subprocess.run(cmd, shell=True)

        files = glob.glob(os.path.join(download_folder, "*"))
        mtlf_files = [f for f in files if 'MTLF' in os.path.basename(f)]


        if not mtlf_files:
            print("No MTLF files found.")
            return

        mtlf_files.sort(key=lambda x: ''.join(filter(str.isdigit, os.path.basename(x))))
        # Now that it's sorted, the LATEST timestamp is the LAST item
        latest_file = mtlf_files[-1]
        print(f"\nLatest file identified: {os.path.basename(latest_file)}")
        print(f"\nLatest file identified: {os.path.basename(latest_file)}")

        process_file(latest_file)


    # Add this to remove the folder once empty
    finally:
        if os.path.exists(download_folder):
            shutil.rmtree(download_folder)
            print(f"Cleanup: Folder {download_folder} has been removed.")

def process_file(file_path):
    global warning_string

    file_name = os.path.basename(file_path)
    print(f"\n--- Processing File: {file_name} ---")

    time_of_file = ''.join(filter(str.isdigit, file_name)) + "00"
    print(f"time of file is {time_of_file}")

    if not os.path.isfile(file_path):
        return

    df = pd.read_csv(file_path)
    df.columns = [c.strip() for c in df.columns]

    actual_columns = set(df.columns)
    expected_columns = ESSENTIAL_ACTUAL_COLUMNS
    known_optional_columns = {'BAA'}

    # Check for column mismatches and handle them.
    if not expected_columns.issubset(actual_columns):
        missing_cols = expected_columns - actual_columns
        exception_str = (
            f"\n[ERROR] File '{file_path}' has schema mismatch. "
            f"Missing essential columns: {missing_cols}. "
            f"Skipping file.\n"
        )
        warning_string += exception_str
        print(exception_str.strip())
        return

    truly_new_cols = actual_columns - expected_columns - known_optional_columns
    if truly_new_cols:
        warning_string += f"\n[INFO] File '{file_path}' has new, unexpected columns: {truly_new_cols}.\n"

    df['IntervalEnd'] = pd.to_datetime(df['Interval'], errors='coerce')
    df['GMTIntervalEnd'] = pd.to_datetime(df['GMTIntervalEnd'], errors='coerce')
    df.dropna(subset=['IntervalEnd', 'GMTIntervalEnd'], inplace=True)

    df['MTLF'] = pd.to_numeric(df.get('MTLF'), errors='coerce').fillna(0)
    df['AveragedActual'] = pd.to_numeric(df.get('Averaged Actual'), errors='coerce').fillna(0)

    df['ForecastTime'] = pd.to_datetime(time_of_file, format='%Y%m%d%H%M%S')

    # Derive baa_zone: 'E' for SPP, 'W' for SWPW.
    # Old-format files that lack the BAA column default to 'E' (SPP only).
    if 'BAA' in df.columns:
        df['baa_zone'] = df['BAA'].str.strip().map(BAA_ZONE_MAP)
        unknown_baa = df['baa_zone'].isna() & df['BAA'].notna()
        if unknown_baa.any():
            bad_vals = df.loc[unknown_baa, 'BAA'].unique().tolist()
            warning_string += (
                f"\n[WARNING] File '{file_path}' contains unrecognised BAA values: {bad_vals}. "
                f"Rows will be skipped.\n"
            )
        df.dropna(subset=['baa_zone'], inplace=True)
    else:
        df['baa_zone'] = 'E'

    df_filtered = df[df['IntervalEnd'].dt.year > 2000].copy()

    if df_filtered.empty:
        print("No valid rows found after filtering.")
        return

    df_filtered['IntervalEnd'] = df_filtered['IntervalEnd'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df_filtered['GMTIntervalEnd'] = df_filtered['GMTIntervalEnd'].dt.strftime('%Y-%m-%d %H:%M:%S')
    df_filtered['ForecastTime'] = df_filtered['ForecastTime'].dt.strftime('%Y-%m-%d %H:%M:%S')

    df_upload = df_filtered[['ForecastTime', 'IntervalEnd', 'GMTIntervalEnd', 'MTLF', 'AveragedActual', 'baa_zone']]
    print(f"Rows found: {len(df_upload)}")

    sql_functions.replace_into_sql_table(
            df=df_upload,
            final_table_fullname='spp_temp.MTLF_mte',
            print_details=True
        )


def update_latest_tables(start_dt,lookback_days=5):

    start_time=start_dt-datetime.timedelta(days=lookback_days)
    sql_functions.run_sql_query("""
        CREATE TABLE IF NOT EXISTS spp_temp.baa_zonal_latest_load_forecast_mte (
         `dt` date NOT NULL,
         `hr` tinyint(3) unsigned NOT NULL,
         `baa_zone` char(10) NOT NULL DEFAULT 'E',
         `load_MW` float(8,2) DEFAULT NULL,
         `actual_load_MW` float(8,2) DEFAULT NULL,
         PRIMARY KEY (`dt`,`hr`,`baa_zone`)
        ) ENGINE=MyISAM
    """)

    query_load_latest = f"""
        REPLACE INTO spp_temp.baa_zonal_latest_load_forecast_mte
        (dt,hr,baa_zone,load_MW,actual_load_MW)
        SELECT date(w.IntervalEnd), hour(w.IntervalEnd)+1, w.baa_zone,
               w.MTLF, NULLIF(w.AveragedActual, 0)
        FROM
        spp_temp.MTLF_mte w
        JOIN (SELECT IntervalEnd, baa_zone, max(ForecastTime) ForecastTime
        FROM spp_temp.MTLF_mte WHERE IntervalEnd > Date('{start_time.date()}') GROUP BY IntervalEnd, baa_zone) ft
            ON w.IntervalEnd=ft.IntervalEnd AND w.baa_zone=ft.baa_zone AND w.ForecastTime=ft.ForecastTime
    """
    sql_functions.run_sql_query(query_load_latest)
    print("\ndata updated in latest table")


if __name__ == "__main__":
    """Main execution function."""
    timezone = pytz.timezone("America/Chicago")
    log_message("Debug", "Information", sys._getframe().f_lineno, "Starting Script\n")
    warning_string= ''

    start_dt_str = sys.argv[1] if len(sys.argv) > 1 else None
    end_dt_str = sys.argv[2] if len(sys.argv) > 2 else None


    sql_functions.run_sql_query("""
            CREATE TABLE IF NOT EXISTS spp_temp.MTLF_mte (
             `ForecastTime` datetime NOT NULL,
             `IntervalEnd` datetime NOT NULL,
             `GMTIntervalEnd` datetime DEFAULT NULL,
             `MTLF` float(10,2) NOT NULL DEFAULT 0.0,
             `AveragedActual` float(10,2) NOT NULL DEFAULT 0.0,
             `baa_zone` char(10) NOT NULL DEFAULT 'E',
             PRIMARY KEY (`ForecastTime`,`IntervalEnd`,`baa_zone`)
            ) ENGINE=MyISAM
        """)

    now = datetime.datetime.now(timezone)
    if start_dt_str:
        try:
            start_dt = pd.to_datetime(start_dt_str, utc=True).tz_convert("US/Central")
        except Exception as e:
            print(f"Error parsing end date: {e}")
        try:
            end_dt = pd.to_datetime(end_dt_str, utc=True).tz_convert("US/Central") if end_dt_str else start_dt
        except Exception as e:
            print(f"Error parsing end date: {e}")
    else:
        start_dt = now
        end_dt = now


    ranges=pd.date_range('2026-03-14','2026-03-27')

    for i in ranges:
        current_dt = i
        download_forecast(current_dt)
        # # Update derived tables
    update_latest_tables(current_dt)




